In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

DATASET_PATH = "synthetic_digits"
IMG_SIZE = 28
BATCH_SIZE = 64
EPOCHS = 10
SEED = 42

tf.random.set_seed(SEED)
images, labels = [], []

for digit_class in range(1, 10):  # 1-9 only, skip 0
    folder = os.path.join(DATASET_PATH, str(digit_class))
    if not os.path.isdir(folder):
        print(f"Warning: folder not found: {folder}")
        continue

    for fname in os.listdir(folder):
        if not fname.lower().endswith(".png"):
            continue

        path = os.path.join(folder, fname)
        img = cv2.imread(path)
        if img is None:
            continue

        img = cv2.bitwise_not(img)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        images.append(img)
        labels.append(digit_class)

if len(images) == 0:
    raise RuntimeError("No images loaded. Check DATASET_PATH and folder structure.")

images = np.array(images, dtype="float32") / 255.0
images = np.expand_dims(images, axis=-1)
labels = np.array(labels, dtype="int32")

print(f"Synthetic (no zeros): {len(images)} images")

# Load MNIST and filter out 0s
(mnist_x_train, mnist_y_train), (mnist_x_test, mnist_y_test) = (
    tf.keras.datasets.mnist.load_data()
)

mnist_images = np.concatenate([mnist_x_train, mnist_x_test], axis=0)
mnist_labels = np.concatenate([mnist_y_train, mnist_y_test], axis=0)

# Filter out 0s from MNIST
mask = mnist_labels != 0
mnist_images = mnist_images[mask]
mnist_labels = mnist_labels[mask]

if IMG_SIZE != 28:
    mnist_images = np.array(
        [cv2.resize(img, (IMG_SIZE, IMG_SIZE)) for img in mnist_images]
    )

mnist_images = mnist_images.astype("float32") / 255.0
mnist_images = np.expand_dims(mnist_images, axis=-1)
mnist_labels = mnist_labels.astype("int32")

print(f"MNIST (no zeros): {len(mnist_images)} images")

# Combine
images = np.concatenate([images, mnist_images], axis=0)
labels = np.concatenate([labels, mnist_labels], axis=0)

print(
    f"Combined: {len(images)} images | shape: {images.shape} | classes: {np.unique(labels)}"
)

# Convert labels 1-9 to 0-8 for the model
labels = labels - 1

x_train, x_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.30, random_state=SEED, stratify=labels
)
x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"Train: {len(x_train)} | Val: {len(x_val)} | Test: {len(x_test)}")


def build_model(input_shape=(IMG_SIZE, IMG_SIZE, 1), num_classes=9):
    model = models.Sequential(
        [
            layers.Conv2D(
                32, (3, 3), padding="same", activation="relu", input_shape=input_shape
            ),
            layers.BatchNormalization(),
            layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),
            layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
            layers.BatchNormalization(),
            layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),
            layers.Flatten(),
            layers.Dense(256, activation="relu"),
            layers.BatchNormalization(),
            layers.Dropout(0.40),
            layers.Dense(num_classes, activation="softmax"),
        ]
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


model = build_model()
model.summary()

cb_list = [
    callbacks.EarlyStopping(
        monitor="val_accuracy", patience=6, restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
]

history = model.fit(
    x_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(x_val, y_val),
    callbacks=cb_list,
)

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nTest Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"], label="Train Acc")
axes[0].plot(history.history["val_accuracy"], label="Val Acc")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Train Loss")
axes[1].plot(history.history["val_loss"], label="Val Loss")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

model.save("digit_model_centered.keras")
print("Training finished (model saved).")